SIB - P7
A Jupyter Notebook with the hyperparameter optimization exercises (grid search, randomized search and tuning the NeuralNetwork).

In [1]:
import numpy as np

from si.data.dataset import Dataset
from si.metrics.accuracy import accuracy
from si.model_selection.grid_search import grid_search_cv
from si.model_selection.randomized_search import randomized_search_cv, log_uniform
from si.model_selection.split import train_test_split
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.layers import DenseLayer
from si.neural_networks.losses import BinaryCrossEntropy
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.optimizers import SGD

## Grid search na NeuralNetwork (antes do Exercício 17)

A `build_network` é a do slide 10: a rede é criada de novo em cada fold, com os mesmos pesos iniciais (`np.random.seed(0)`).

In [2]:
def build_network(learning_rate, n_hidden):
    np.random.seed(0)  # same initial weights for every configuration
    net = NeuralNetwork(epochs=100, batch_size=16, optimizer=SGD,
                        learning_rate=learning_rate, momentum=0.9,
                        loss=BinaryCrossEntropy, metric=accuracy)
    net.add(DenseLayer(n_hidden, (2,)))
    net.add(ReLUActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    return net

rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, size=(400, 2))
y = (X[:, 0]**2 + X[:, 1]**2 < 1.5).astype(float)  # inside the circle?
ds = Dataset(X, y)

grid = {'learning_rate': [0.001, 0.01, 0.1], 'n_hidden': [2, 4, 16]}
results = grid_search_cv(build_network, ds, grid, cv=5, seed=1)
for h, s in zip(results['hyperparameters'], results['scores']):
    print(h, round(s, 3))
print(results['best_hyperparameters'], results['best_score'])
print("Proporção da classe 0:", round(1 - y.mean(), 3))

{'learning_rate': 0.001, 'n_hidden': 2} 0.715
{'learning_rate': 0.001, 'n_hidden': 4} 0.715
{'learning_rate': 0.001, 'n_hidden': 16} 0.84
{'learning_rate': 0.01, 'n_hidden': 2} 0.707
{'learning_rate': 0.01, 'n_hidden': 4} 0.732
{'learning_rate': 0.01, 'n_hidden': 16} 0.977
{'learning_rate': 0.1, 'n_hidden': 2} 0.717
{'learning_rate': 0.1, 'n_hidden': 4} 0.753
{'learning_rate': 0.1, 'n_hidden': 16} 0.982
{'learning_rate': 0.1, 'n_hidden': 16} 0.9824999999999999
Proporção da classe 0: 0.715


| Pergunta | O que observo |
|---|---|
| Qual é a melhor configuração e o seu score? | `learning_rate=0.1, n_hidden=16`, com accuracy média de **0.982**. Com 16 unidades, `lr=0.01` fica muito perto (0.977). |
| Porque é que várias configurações dão 0.715? | 71.5% das amostras estão fora do círculo (classe 0). Uma rede que prevê sempre 0 acerta exatamente 0.715. Com poucas unidades (2–4) e/ou `lr` pequeno, a rede não consegue aprender a fronteira circular e fica presa na classe maioritária. |
| Quantos modelos foram treinados no total? | 9 configurações × 5 folds = **45 modelos**. |

In [3]:
# Passar um modelo já construído em vez da função: copy.deepcopy + setattr
results_model = grid_search_cv(build_network(0.01, 16), ds, {'learning_rate': [0.0001, 0.1]}, cv=5, seed=1)
for h, s in zip(results_model['hyperparameters'], results_model['scores']):
    print(h, round(s, 3))

{'learning_rate': 0.0001} 0.978
{'learning_rate': 0.1} 0.975


**Os scores mudam?** Praticamente não (0.978 e 0.975, os dois perto do 0.977 de `lr=0.01, n_hidden=16`). O `setattr` cria um atributo `learning_rate` na rede, mas o otimizador já foi criado no `__init__` com `lr=0.01` e copiado para as camadas pelo `add`, por isso o valor novo nunca é usado. A pequena diferença entre os dois vem só da aleatoriedade do shuffle dos mini-batches. É por isso que, para a NeuralNetwork, se passa uma **função builder**.

## Exercício 17 – `randomized_search_cv` e `log_uniform`

Implementados em `src/si/model_selection/randomized_search.py` (os folds e a validação são partilhados com `grid_search.py`). Verificação rápida do `log_uniform` contra o `uniform`:

In [4]:
rng = np.random.default_rng(0)
sampler = log_uniform(1e-4, 1e-1)
log_samples = np.array([sampler(rng) for _ in range(10000)])
uni_samples = rng.uniform(1e-4, 1e-1, 10000)
for name, s in [("uniform", uni_samples), ("log_uniform", log_samples)]:
    print(f"{name:12s} fração em [1e-4,1e-3): {np.mean(s < 1e-3):.2f} | [1e-3,1e-2): {np.mean((s >= 1e-3) & (s < 1e-2)):.2f} | [1e-2,1e-1]: {np.mean(s >= 1e-2):.2f}")

uniform      fração em [1e-4,1e-3): 0.01 | [1e-3,1e-2): 0.09 | [1e-2,1e-1]: 0.90
log_uniform  fração em [1e-4,1e-3): 0.33 | [1e-3,1e-2): 0.33 | [1e-2,1e-1]: 0.34


## Exercício 18 – afinar a rede do Exercício 16

Dataset: **breast_bin** (já está em `datasets/`), com 699 amostras e 9 features. Cada feature é uma característica das células, com valor de 1 a 10. O label diz se o tumor é benigno (0) ou maligno (1). Divido as features por 10 para ficarem entre 0 e 1, o que ajuda a rede a treinar de forma estável.

Arquitetura igual à do Exercício 16, mas com 9 entradas: 9 → 16 → 8 → 1. A pesquisa usa só o treino; o teste é usado uma vez, no fim.

In [5]:
from si.io.csv_file import read_csv

dataset = read_csv('../datasets/breast_bin/breast-bin.csv', features=False, label=True)
dataset.X = dataset.X / 10  # valores de 1 a 10 -> 0.1 a 1
print("Amostras e features:", dataset.X.shape, "| malignos:", int(dataset.y.sum()))

train, test = train_test_split(dataset, test_size=0.2, random_state=42)

def build_network(learning_rate, batch_size, epochs):
    np.random.seed(0)  # same initial weights for every configuration
    net = NeuralNetwork(epochs=epochs, batch_size=batch_size, optimizer=SGD, learning_rate=learning_rate,
                        loss=BinaryCrossEntropy, metric=accuracy)
    net.add(DenseLayer(16, (9,)))
    net.add(ReLUActivation())
    net.add(DenseLayer(8))
    net.add(ReLUActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    return net

distributions = {'learning_rate': log_uniform(1e-4, 1e-1),
                 'batch_size': [8, 16, 32, 64],
                 'epochs': [50, 100, 200]}
results = randomized_search_cv(build_network, train, distributions, scoring=accuracy, cv=3, n_iter=20, seed=42)

order = np.argsort(results['scores'])[::-1]
for i in order:
    h = results['hyperparameters'][i]
    print(f"lr={h['learning_rate']:.5f} batch={h['batch_size']:2d} epochs={h['epochs']:3d} -> {results['scores'][i]:.3f}")
print("Melhor:", results['best_hyperparameters'], round(results['best_score'], 3))

Amostras e features: (699, 9) | malignos: 241


lr=0.00461 batch=64 epochs= 50 -> 0.968
lr=0.00024 batch=64 epochs=100 -> 0.968
lr=0.00130 batch= 8 epochs=200 -> 0.966
lr=0.00854 batch=16 epochs=200 -> 0.966
lr=0.00029 batch=32 epochs=200 -> 0.964
lr=0.00019 batch=32 epochs=200 -> 0.964
lr=0.01921 batch=32 epochs=200 -> 0.964
lr=0.02098 batch=32 epochs=100 -> 0.964
lr=0.01880 batch=32 epochs=100 -> 0.964
lr=0.02163 batch=64 epochs= 50 -> 0.964
lr=0.00025 batch=32 epochs=100 -> 0.963
lr=0.00256 batch=64 epochs= 50 -> 0.963
lr=0.01715 batch=16 epochs=200 -> 0.961
lr=0.03765 batch= 8 epochs=200 -> 0.961
lr=0.00048 batch=32 epochs=200 -> 0.961
lr=0.00214 batch=16 epochs= 50 -> 0.961
lr=0.00251 batch=16 epochs= 50 -> 0.961
lr=0.03040 batch=16 epochs=100 -> 0.961
lr=0.08168 batch=16 epochs=200 -> 0.959
lr=0.00095 batch=64 epochs=100 -> 0.957
Melhor: {'learning_rate': 0.0046105672499844215, 'batch_size': 64, 'epochs': 50} 0.968


In [6]:
# Refit da melhor configuração em todo o treino e avaliação no teste
best = build_network(**results['best_hyperparameters'])
best.fit(train)
best_test = accuracy(test.y, best.predict(test))

# Hiperparâmetros do Exercício 16, avaliados da mesma forma
baseline_cfg = {'learning_rate': 0.01, 'batch_size': 16, 'epochs': 100}
baseline_cv = randomized_search_cv(build_network, train, {k: [v] for k, v in baseline_cfg.items()},
                                   scoring=accuracy, cv=3, n_iter=1, seed=42)['best_score']
baseline = build_network(**baseline_cfg)
baseline.fit(train)
baseline_test = accuracy(test.y, baseline.predict(test))

print(f"Afinada      CV: {results['best_score']:.3f} | teste: {best_test:.3f}")
print(f"Exercício 16 CV: {baseline_cv:.3f} | teste: {baseline_test:.3f}")

Afinada      CV: 0.968 | teste: 0.957
Exercício 16 CV: 0.964 | teste: 0.942


**A afinação ajudou?** Um pouco.

| | CV (média de 3 folds) | Teste |
|---|---|---|
| Afinada (`lr≈0.0046, batch=64, epochs=50`) | 0.968 | **0.957** |
| Exercício 16 (`lr=0.01, batch=16, epochs=100`) | 0.964 | 0.942 |

- No teste, a rede afinada acerta cerca de 1.5 pontos a mais, o que corresponde a uns 2 tumores a mais bem classificados em 140.
- No CV, todas as 20 configurações ficam entre 0.957 e 0.968. O problema é fácil e a rede acerta bem com quase qualquer configuração razoável, por isso a diferença é pequena e parte dela pode ser sorte do split.
- A melhor configuração também é a mais barata: 50 épocas em vez de 100.
- Custo da pesquisa: 20 configurações × 3 folds = 60 treinos, mais 1 refit.